# RAG Evaluation Suite

Quantitative evaluation of the InsureLLM agentic RAG pipeline across **15 ground-truth test cases** spanning all four knowledge-base categories (*employees*, *contracts*, *company*, *products*).

## Metrics

| Category | Metric | Description |
|---|---|---|
| Retrieval | **Hit Rate\@3** | Correct source document appears in top-3 retrieved chunks |
| Retrieval | **MRR** | Mean Reciprocal Rank of the first relevant result |
| Retrieval (LLM judge) | **Context Precision** | Fraction of retrieved chunks relevant to the query |
| Retrieval (LLM judge) | **Context Recall** | Retrieved context covers all information needed to answer |
| Generation (LLM judge) | **Faithfulness** | Answer is grounded in context — no hallucinated claims |
| Generation (LLM judge) | **Answer Relevancy** | Answer directly and completely addresses the question |
| Generation (LLM judge) | **Answer Correctness** | Answer matches the ground-truth facts |
| Agentic | **Tool Selection Accuracy** | Agent invoked the correct retrieval tool(s) |
| Agentic | **Tool Efficiency** | Average tool calls and LLM iterations per question |


In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


In [2]:
load_dotenv(override=True)

NANO = "gpt-4.1-nano"
openai_client = OpenAI()

CHROMA_PATH = "chroma_db"
DOC_TYPES = ["employees", "contracts", "company", "products"]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstores = {}
for doc_type in DOC_TYPES:
    vectorstores[doc_type] = Chroma(
        collection_name=doc_type,
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH,
    )
    count = vectorstores[doc_type]._collection.count()
    print(f"  {doc_type}: {count} chunks")

print("Vectorstores loaded.")


  employees: 118 chunks
  contracts: 230 chunks
  company: 18 chunks
  products: 47 chunks
Vectorstores loaded.


In [ ]:
# ── Retrieval helpers ──────────────────────────────────────────────────
def _retrieve_raw(doc_type, query, k=3):
    retriever = vectorstores[doc_type].as_retriever(search_kwargs={'k': k})
    return retriever.invoke(query)


def _retrieve_fmt(doc_type, query, k=3):
    docs = _retrieve_raw(doc_type, query, k)
    if not docs:
        return f"No results for {doc_type}."
    sep = "\n\n---\n\n"
    return sep.join(
        f"[{doc_type.upper()} | {r.metadata.get('source', 'unknown')}]\n{r.page_content}"
        for r in docs
    )


def search_employees(query): return _retrieve_fmt('employees', query)
def search_contracts(query): return _retrieve_fmt('contracts', query)
def search_company(query):   return _retrieve_fmt('company', query)
def search_products(query):  return _retrieve_fmt('products', query)

TOOL_FUNCTIONS = {
    'search_employees': search_employees,
    'search_contracts': search_contracts,
    'search_company':   search_company,
    'search_products':  search_products,
}


# ── Tool schemas ────────────────────────────────────────────────────────
def _tool(name, desc):
    return {
        "type": "function",
        "function": {
            "name": name, "description": desc,
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query."}},
                "required": ["query"],
            },
        },
    }


tools = [
    _tool("search_employees", "Search HR records: staff roles, salaries, performance, career history."),
    _tool("search_contracts", "Search client contracts: pricing, SLAs, duration, renewal terms."),
    _tool("search_company",   "Search company info: overview, mission, headcount, structure."),
    _tool("search_products",  "Search product docs: features, pricing tiers, roadmap."),
]


# ── System prompt ───────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a helpful InsureLLM assistant.\n"
    "Always use the appropriate tool(s) before answering.\n"
    "Base answers strictly on retrieved content."
)


# ── Instrumented agent — logs all tool calls and retrieved context ───────
def eval_agent(question):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    log = []
    for i in range(10):
        resp = openai_client.chat.completions.create(
            model=NANO, messages=msgs, tools=tools, tool_choice='auto'
        )
        choice = resp.choices[0]
        msgs.append(choice.message)
        if choice.finish_reason == "tool_calls":
            for tc in choice.message.tool_calls:
                fn = tc.function.name
                args = json.loads(tc.function.arguments)
                result = TOOL_FUNCTIONS[fn](**args) if fn in TOOL_FUNCTIONS else f'Unknown: {fn}'
                log.append({'name': fn, 'query': args.get('query', ''), 'result': result})
                msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            return {
                "answer": choice.message.content,
                "tool_calls": log,
                "iterations": i + 1,
                "context": "\n\n".join(t["result"] for t in log),
            }
    return {
        "answer": "Max iterations reached.",
        "tool_calls": log,
        "iterations": 10,
        "context": "\n\n".join(t["result"] for t in log),
    }


print("Helpers, tools, and instrumented agent ready.")


In [ ]:
# 15 ground-truth test cases.
# expected_source_keywords: substrings that should appear in the source filename.
# expected_tools: retrieval tool(s) the agent should call.
TEST_CASES = [
    # ── Employees (5) ─────────────────────────────────────────────────
    {"id": "emp_01",
     "question": "What is Robert Chen's current salary?",
     "ground_truth": "Robert Chen's current salary is $152,000.",
     "expected_tools": ["search_employees"],
     "expected_source_keywords": ["Robert Chen"], "category": "employees"},

    {"id": "emp_02",
     "question": "What was Robert Chen's performance rating in 2023?",
     "ground_truth": "Robert Chen received a performance rating of 4.8 out of 5 in 2023.",
     "expected_tools": ["search_employees"],
     "expected_source_keywords": ["Robert Chen"], "category": "employees"},

    {"id": "emp_03",
     "question": "What is Priya Sharma's job title?",
     "ground_truth": "Priya Sharma's job title is Senior Data Scientist.",
     "expected_tools": ["search_employees"],
     "expected_source_keywords": ["Priya Sharma"], "category": "employees"},

    {"id": "emp_04",
     "question": "Where did Priya Sharma complete her PhD?",
     "ground_truth": "Priya Sharma completed her PhD at Stanford University.",
     "expected_tools": ["search_employees"],
     "expected_source_keywords": ["Priya Sharma"], "category": "employees"},

    {"id": "emp_05",
     "question": "How many engineers does Robert Chen currently mentor?",
     "ground_truth": "Robert Chen currently mentors a team of 6 engineers.",
     "expected_tools": ["search_employees"],
     "expected_source_keywords": ["Robert Chen"], "category": "employees"},

    # ── Contracts (3) ─────────────────────────────────────────────────
    {"id": "con_01",
     "question": "What is the monthly payment in the Apex Reinsurance contract for Rellm?",
     "ground_truth": "The monthly payment in the Apex Reinsurance contract is $10,000.",
     "expected_tools": ["search_contracts"],
     "expected_source_keywords": ["Apex Reinsurance"], "category": "contracts"},

    {"id": "con_02",
     "question": "What is the duration of the Apex Reinsurance contract for Rellm?",
     "ground_truth": "The Apex Reinsurance contract has a duration of 12 months.",
     "expected_tools": ["search_contracts"],
     "expected_source_keywords": ["Apex Reinsurance"], "category": "contracts"},

    {"id": "con_03",
     "question": "How many staff can receive onboarding training under the Apex Reinsurance contract?",
     "ground_truth": "Up to 10 members of the client's staff can receive onboarding training.",
     "expected_tools": ["search_contracts"],
     "expected_source_keywords": ["Apex Reinsurance"], "category": "contracts"},

    # ── Company (3) ───────────────────────────────────────────────────
    {"id": "comp_01",
     "question": "How many employees does InsureLLM have?",
     "ground_truth": "InsureLLM has 32 employees.",
     "expected_tools": ["search_company"],
     "expected_source_keywords": ["overview"], "category": "company"},

    {"id": "comp_02",
     "question": "When was InsureLLM founded?",
     "ground_truth": "InsureLLM was founded in 2015.",
     "expected_tools": ["search_company"],
     "expected_source_keywords": ["overview"], "category": "company"},

    {"id": "comp_03",
     "question": "How many insurance software products does InsureLLM offer?",
     "ground_truth": "InsureLLM offers 8 insurance software products.",
     "expected_tools": ["search_company"],
     "expected_source_keywords": ["overview"], "category": "company"},

    # ── Products (4) ──────────────────────────────────────────────────
    {"id": "prod_01",
     "question": "What is the basic plan monthly price for Rellm?",
     "ground_truth": "The basic plan for Rellm costs $5,000 per month.",
     "expected_tools": ["search_products"],
     "expected_source_keywords": ["Rellm"], "category": "products"},

    {"id": "prod_02",
     "question": "What is the professional plan monthly price for Rellm?",
     "ground_truth": "The professional plan for Rellm costs $10,000 per month.",
     "expected_tools": ["search_products"],
     "expected_source_keywords": ["Rellm"], "category": "products"},

    {"id": "prod_03",
     "question": "What is the basic tier starting price for Homellm?",
     "ground_truth": "The basic tier for Homellm starts at $5,000 per month.",
     "expected_tools": ["search_products"],
     "expected_source_keywords": ["Homellm"], "category": "products"},

    {"id": "prod_04",
     "question": "What AI risk management features does Rellm provide?",
     "ground_truth": "Rellm provides AI-driven analytics for predictive risk insights, real-time data analysis, and a risk assessment module using historical data and advanced modeling.",
     "expected_tools": ["search_products"],
     "expected_source_keywords": ["Rellm"], "category": "products"},
]

print(f"Test dataset: {len(TEST_CASES)} cases across {len(set(tc['category'] for tc in TEST_CASES))} categories.")


In [ ]:
# Run direct retrieval (for retrieval metrics) AND full agent (for generation/agentic metrics)
# for every test case. Cache all results — subsequent cells only compute metrics, no extra API calls.

print(f"Running {len(TEST_CASES)} evaluations...\n")
eval_results = []

for i, tc in enumerate(TEST_CASES, 1):
    print(f"[{i:2d}/{len(TEST_CASES)}] {tc['id']}: {tc['question'][:65]}...")

    # Direct retrieval using the expected collection (measures retrieval quality)
    expected_coll = tc['expected_tools'][0].replace('search_', '')
    raw_docs = _retrieve_raw(expected_coll, tc['question'], k=3)

    # Full agent run (measures generation and agentic quality)
    agent_out = eval_agent(tc['question'])

    eval_results.append({'test_case': tc, 'raw_docs': raw_docs, 'agent_result': agent_out})
    called = [t['name'] for t in agent_out['tool_calls']]
    print(f"        tools={called}  iterations={agent_out['iterations']}")

print(f"\nDone. {len(eval_results)} evaluations complete.")


In [ ]:
# ── Retrieval metrics (no LLM calls) ───────────────────────────────────

def _matches(doc, keywords):
    src = Path(doc.metadata.get('source', '')).name
    return any(kw.lower() in src.lower() for kw in keywords)


def compute_hit_rate(results):
    hits = []
    for r in results:
        tc = r['test_case']
        hit = any(_matches(d, tc['expected_source_keywords']) for d in r['raw_docs'])
        srcs = [Path(d.metadata.get('source', '')).name for d in r['raw_docs']]
        hits.append({'id': tc['id'], 'hit': hit, 'sources': srcs})
    rate = sum(h['hit'] for h in hits) / len(hits) if hits else 0
    return {'hit_rate': rate, 'details': hits}


def compute_mrr(results):
    rrs = []
    for r in results:
        tc = r['test_case']
        rr = 0.0
        for rank, doc in enumerate(r['raw_docs'], 1):
            if _matches(doc, tc['expected_source_keywords']):
                rr = 1.0 / rank
                break
        rrs.append({'id': tc['id'], 'rr': rr})
    mrr = sum(r['rr'] for r in rrs) / len(rrs) if rrs else 0
    return {'mrr': mrr, 'details': rrs}


hr  = compute_hit_rate(eval_results)
mrr = compute_mrr(eval_results)

print(f"Hit Rate@3 : {hr['hit_rate']:.2%}")
print(f"MRR        : {mrr['mrr']:.4f}")
print()
print("Per-question details:")
for d in hr['details']:
    status = 'HIT ' if d['hit'] else 'MISS'
    print(f"  [{status}] {d['id']:8s}  sources={d['sources']}")


In [ ]:
# ── LLM-as-Judge helper ─────────────────────────────────────────────────
# Uses gpt-4.1-nano to score 0.0–1.0 with brief reasoning.

def llm_judge(prompt):
    resp = openai_client.chat.completions.create(
        model=NANO,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise RAG evaluator. "
                    "Always respond with valid JSON containing exactly two keys: "
                    "'score' (float 0.0–1.0) and 'reasoning' (one sentence)."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    try:
        data = json.loads(resp.choices[0].message.content)
        return {'score': float(data.get('score', 0.0)), 'reasoning': data.get('reasoning', '')}
    except (json.JSONDecodeError, ValueError):
        return {'score': 0.0, 'reasoning': 'Failed to parse judge response.'}


print("LLM judge ready.")


In [ ]:
# ── Context Precision ───────────────────────────────────────────────────
# Of the retrieved chunks, what fraction are relevant to the query?

def compute_context_precision(results):
    scores = []
    for r in results:
        tc = r['test_case']
        docs = r['raw_docs']
        if not docs:
            scores.append({'id': tc['id'], 'score': 0.0, 'chunk_scores': []})
            continue
        chunk_scores = []
        for doc in docs:
            j = llm_judge(
                "Evaluate whether this retrieved chunk is relevant to answering the question.\n\n"
                f"Question: {tc['question']}\n\n"
                f"Chunk:\n{doc.page_content[:600]}\n\n"
                "Score 1.0 = directly useful for answering.\n"
                "Score 0.5 = partially relevant.\n"
                "Score 0.0 = irrelevant."
            )
            chunk_scores.append(j['score'])
        avg = sum(chunk_scores) / len(chunk_scores)
        scores.append({'id': tc['id'], 'score': avg, 'chunk_scores': chunk_scores})
    overall = sum(s['score'] for s in scores) / len(scores) if scores else 0
    return {'context_precision': overall, 'details': scores}


print("Computing Context Precision (LLM judge)...")
ctx_precision = compute_context_precision(eval_results)
print(f"Context Precision: {ctx_precision['context_precision']:.2%}")
for d in ctx_precision['details']:
    print(f"  {d['id']:8s}  avg={d['score']:.2f}  chunks={[round(s,2) for s in d['chunk_scores']]}") 


In [ ]:
# ── Context Recall ──────────────────────────────────────────────────────
# Does the retrieved context contain ALL the information needed to answer?

def compute_context_recall(results):
    scores = []
    for r in results:
        tc = r['test_case']
        context = '\n\n---\n\n'.join(d.page_content for d in r['raw_docs'])
        j = llm_judge(
            "Evaluate whether the retrieved context contains sufficient information to answer the question.\n\n"
            f"Question: {tc['question']}\n"
            f"Expected answer: {tc['ground_truth']}\n\n"
            f"Retrieved context (may be truncated):\n{context[:2500]}\n\n"
            "Score 1.0 = context fully covers all facts needed for the expected answer.\n"
            "Score 0.5 = context partially covers needed information.\n"
            "Score 0.0 = context is missing key information."
        )
        scores.append({'id': tc['id'], 'score': j['score'], 'reasoning': j['reasoning']})
    overall = sum(s['score'] for s in scores) / len(scores) if scores else 0
    return {'context_recall': overall, 'details': scores}


print("Computing Context Recall (LLM judge)...")
ctx_recall = compute_context_recall(eval_results)
print(f"Context Recall: {ctx_recall['context_recall']:.2%}")
for d in ctx_recall['details']:
    print(f"  {d['id']:8s}  score={d['score']:.2f}  | {d['reasoning'][:80]}")


In [ ]:
# ── Faithfulness ────────────────────────────────────────────────────────
# Is every claim in the generated answer supported by the retrieved context?
# Measures hallucination — a score < 1 means ungrounded claims were made.

def compute_faithfulness(results):
    scores = []
    for r in results:
        tc = r['test_case']
        ag = r['agent_result']
        if not ag['context']:
            scores.append({'id': tc['id'], 'score': 0.0, 'reasoning': 'No context retrieved.'})
            continue
        j = llm_judge(
            "Evaluate whether the generated answer is fully supported by the retrieved context.\n\n"
            f"Question: {tc['question']}\n\n"
            f"Retrieved context (may be truncated):\n{ag['context'][:2500]}\n\n"
            f"Generated answer:\n{ag['answer']}\n\n"
            "Score 1.0 = every claim in the answer is directly supported by the context.\n"
            "Score 0.5 = most claims supported but minor extrapolation present.\n"
            "Score 0.0 = answer contains significant claims not found in context."
        )
        scores.append({'id': tc['id'], 'score': j['score'], 'reasoning': j['reasoning']})
    overall = sum(s['score'] for s in scores) / len(scores) if scores else 0
    return {'faithfulness': overall, 'details': scores}


print("Computing Faithfulness (LLM judge)...")
faithfulness = compute_faithfulness(eval_results)
print(f"Faithfulness: {faithfulness['faithfulness']:.2%}")
for d in faithfulness['details']:
    print(f"  {d['id']:8s}  score={d['score']:.2f}  | {d['reasoning'][:80]}")


In [ ]:
# ── Answer Relevancy ────────────────────────────────────────────────────
# Does the generated answer directly and completely address the question?

def compute_answer_relevancy(results):
    scores = []
    for r in results:
        tc = r['test_case']
        ag = r['agent_result']
        j = llm_judge(
            "Evaluate whether the answer directly and completely addresses the question.\n\n"
            f"Question: {tc['question']}\n\n"
            f"Answer:\n{ag['answer']}\n\n"
            "Score 1.0 = answer fully and directly addresses the question.\n"
            "Score 0.5 = answer is partially relevant or misses some aspect.\n"
            "Score 0.0 = answer does not address the question."
        )
        scores.append({'id': tc['id'], 'score': j['score'], 'reasoning': j['reasoning']})
    overall = sum(s['score'] for s in scores) / len(scores) if scores else 0
    return {'answer_relevancy': overall, 'details': scores}


print("Computing Answer Relevancy (LLM judge)...")
answer_relevancy = compute_answer_relevancy(eval_results)
print(f"Answer Relevancy: {answer_relevancy['answer_relevancy']:.2%}")
for d in answer_relevancy['details']:
    print(f"  {d['id']:8s}  score={d['score']:.2f}  | {d['reasoning'][:80]}")


In [ ]:
# ── Answer Correctness ──────────────────────────────────────────────────
# Does the generated answer match the ground-truth facts?

def compute_answer_correctness(results):
    scores = []
    for r in results:
        tc = r['test_case']
        ag = r['agent_result']
        j = llm_judge(
            "Evaluate whether the generated answer matches the ground truth in factual accuracy.\n\n"
            f"Question: {tc['question']}\n"
            f"Ground truth: {tc['ground_truth']}\n\n"
            f"Generated answer:\n{ag['answer']}\n\n"
            "Score 1.0 = answer contains all key facts from the ground truth.\n"
            "Score 0.5 = answer is partially correct (some facts right, some missing or wrong).\n"
            "Score 0.0 = answer is incorrect or contradicts the ground truth."
        )
        scores.append({'id': tc['id'], 'score': j['score'], 'reasoning': j['reasoning']})
    overall = sum(s['score'] for s in scores) / len(scores) if scores else 0
    return {'answer_correctness': overall, 'details': scores}


print("Computing Answer Correctness (LLM judge)...")
answer_correctness = compute_answer_correctness(eval_results)
print(f"Answer Correctness: {answer_correctness['answer_correctness']:.2%}")
for d in answer_correctness['details']:
    print(f"  {d['id']:8s}  score={d['score']:.2f}  | {d['reasoning'][:80]}")


In [ ]:
# ── Agentic Metrics ─────────────────────────────────────────────────────

def compute_tool_accuracy(results):
    rows = []
    for r in results:
        tc = r['test_case']
        called = {t['name'] for t in r['agent_result']['tool_calls']}
        expected = set(tc['expected_tools'])
        rows.append({
            'id': tc['id'], 'expected': sorted(expected),
            'called': sorted(called), 'correct': expected.issubset(called),
        })
    acc = sum(r['correct'] for r in rows) / len(rows) if rows else 0
    return {'tool_accuracy': acc, 'details': rows}


def compute_tool_efficiency(results):
    rows = [{'id': r['test_case']['id'],
             'iterations': r['agent_result']['iterations'],
             'tool_calls': len(r['agent_result']['tool_calls'])}
            for r in results]
    avg_iter  = sum(r['iterations']  for r in rows) / len(rows) if rows else 0
    avg_calls = sum(r['tool_calls']  for r in rows) / len(rows) if rows else 0
    return {'avg_iterations': avg_iter, 'avg_tool_calls': avg_calls, 'details': rows}


tool_accuracy   = compute_tool_accuracy(eval_results)
tool_efficiency = compute_tool_efficiency(eval_results)

print(f"Tool Selection Accuracy : {tool_accuracy['tool_accuracy']:.2%}")
print(f"Avg Iterations/Question : {tool_efficiency['avg_iterations']:.2f}")
print(f"Avg Tool Calls/Question : {tool_efficiency['avg_tool_calls']:.2f}")
print()
print("Per-question tool selection:")
for d in tool_accuracy['details']:
    status = 'OK  ' if d['correct'] else 'FAIL'
    print(f"  [{status}] {d['id']:8s}  expected={d['expected']}  called={d['called']}")


In [ ]:
# ── Summary Report ──────────────────────────────────────────────────────
W = [28, 10, 55]

rows = [
    ("Hit Rate@3",              f"{hr['hit_rate']:.2%}",                          "Correct source doc in top-3 retrieved chunks"),
    ("MRR",                     f"{mrr['mrr']:.4f}",                              "Mean Reciprocal Rank of first relevant result"),
    ("Context Precision",       f"{ctx_precision['context_precision']:.2%}",      "Fraction of retrieved chunks relevant to query"),
    ("Context Recall",          f"{ctx_recall['context_recall']:.2%}",            "Context covers all info needed to answer"),
    ("Faithfulness",            f"{faithfulness['faithfulness']:.2%}",            "Answer grounded in context (no hallucination)"),
    ("Answer Relevancy",        f"{answer_relevancy['answer_relevancy']:.2%}",    "Answer directly addresses the question"),
    ("Answer Correctness",      f"{answer_correctness['answer_correctness']:.2%}","Answer matches ground-truth facts"),
    ("Tool Selection Accuracy", f"{tool_accuracy['tool_accuracy']:.2%}",          "Agent called the correct retrieval tool(s)"),
    ("Avg Tool Calls / Q",      f"{tool_efficiency['avg_tool_calls']:.2f}",       "Average tool calls per question"),
    ("Avg Iterations / Q",      f"{tool_efficiency['avg_iterations']:.2f}",       "Average LLM iterations per question"),
]

sep    = '-' * (sum(W) + 4)
header = f"{'Metric':<{W[0]}} {'Score':>{W[1]}}  {'Description':<{W[2]}}"

print()
print('=' * (sum(W) + 4))
print('  RAG EVALUATION RESULTS')
print('=' * (sum(W) + 4))
print(header)
print(sep)
for metric, score, desc in rows:
    print(f'{metric:<{W[0]}} {score:>{W[1]}}  {desc:<{W[2]}}')
print(sep)
print(f"  Test cases: {len(TEST_CASES)}  |  Model: {NANO}")
print('=' * (sum(W) + 4))

# ── Per-category answer correctness breakdown ────────────────────────────
id_to_cat = {r['test_case']['id']: r['test_case']['category'] for r in eval_results}
categories = sorted(set(id_to_cat.values()))
print("\nPer-category Answer Correctness:")
for cat in categories:
    cat_scores = [d['score'] for d in answer_correctness['details']
                  if id_to_cat.get(d['id']) == cat]
    avg = sum(cat_scores) / len(cat_scores) if cat_scores else 0
    print(f"  {cat:<12s}: {avg:.2%}  ({len(cat_scores)} cases)")
